# Képek manuális dobozolása

In [ ]:
import cv2
import os
from ultralytics import YOLO

def process_fixed_rois(input_path, output_path, rois, model_path='best.pt', conf_threshold=0.30):
    """
    FIX REKESZ (ROI) ALAPÚ FELDOLGOZÁS:
    Csak az előre megadott területeket vizsgálja, és rekeszenként csak 1 dobozt rajzol.
    """
    if not os.path.exists(input_path):
        print(f"Hiba: Nem találom a képet: {input_path}")
        return

    # Sima YOLO betöltése
    model = YOLO(model_path)
    frame = cv2.imread(input_path)
    print(f"Bemeneti kép (Fix ROI): {input_path}")

    # Színek a kategóriákhoz
    colors = {
        "low": (0, 0, 255),
        "medium": (0, 165, 255),
        "high": (0, 255, 0),
        "empty": (128, 128, 128)
    }

    # Végigmegyünk az előre megadott rekeszeken
    for roi_name, (x1, y1, x2, y2) in rois.items():
        
        # Kivágás
        y_min, y_max = min(y1, y2), max(y1, y2)
        x_min, x_max = min(x1, x2), max(x1, x2)

        crop = frame[y_min:y_max, x_min:x_max]

        # YOLO
        results = model(crop, conf=conf_threshold, verbose=False)

        best_class_name = "empty"
        highest_conf = 0.0

        # Kiválasztjuk a legbiztosabbat
        if results[0].boxes is not None and len(results[0].boxes) > 0:
            best_box = sorted(results[0].boxes, key=lambda x: x.conf[0].item(), reverse=True)[0]
            class_id = int(best_box.cls[0].item())
            
            highest_conf = best_box.conf[0].item()
            best_class_name = model.names[class_id] # 'low', 'medium', vagy 'high'

        # Rajzolás
        color = colors.get(best_class_name, (255, 255, 255))
        
        # Megrajzoljuk a fix keretet
        cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), color, 4)
        
        # Felirat hozzáadása a dobozhoz
        if highest_conf > 0:
            label = f"{roi_name}: {best_class_name} ({highest_conf:.2f})"
        else:
            label = f"{roi_name}: Unrecognized"
            
        cv2.putText(frame, label, (x_min + 5, y_max - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    # Kép mentése
    cv2.imwrite(output_path, frame)
    print(f"Az új fix rekeszes kép elmentve: {output_path}\n")

In [ ]:
import cv2
import os
from ultralytics import YOLO

def process_video_fixed_rois(input_path, output_path, rois, model_path='best.pt', conf_threshold=0.30, frame_proc=1):
    """
    VIDEÓ FELDOLGOZÁSA FIX REKESZEKKEL:
    Képkockánként halad, csak a megadott területeket vizsgálja, és új videóként menti.
    """
    if not os.path.exists(input_path):
        print(f"Hiba: Nem találom a bemeneti videót: {input_path}")
        return

    model = YOLO(model_path)
    video_capture = cv2.VideoCapture(input_path)
    video_capture.set(cv2.CAP_PROP_ORIENTATION_AUTO, 1.0)

    # Fájl elérhetősége
    success, first_frame = video_capture.read()
    if not success: 
        print(f"Hiba: A videó üres, vagy nem sikerült megnyitni: {input_path}")
        return
        
    frame_height, frame_width = first_frame.shape[:2]
    fps = int(video_capture.get(cv2.CAP_PROP_FPS)) or 30

    # Visszatekerjük a videót az elejére
    video_capture.set(cv2.CAP_PROP_POS_FRAMES, 0)

    print(f"Bemeneti videó: {input_path}")

    video_writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_width, frame_height))

    # Színek a kategóriákhoz
    colors = {"low": (0, 0, 255),
              "medium": (0, 165, 255),
              "high": (0, 255, 0),
              "empty": (128, 128, 128)}

    frame_count = 0
    process_every = frame_proc
    
    # Utolsó ismert eredmény
    last_predictions = {}

    # Képkockánkénti feldolgozás
    while True:
        ret, frame = video_capture.read()
        if not ret: 
            break  # Ha nincs több képkocka, kilép

        frame_count += 1
        
        # Futtatandó képkocka
        run_yolo = (frame_count % process_every == 1) 

        # Végigmegyünk az előre megadott rekeszeken
        for roi_name, (x1, y1, x2, y2) in rois.items():
            
            # Koordináták normalizálása
            x_min, x_max = max(0, min(x1, x2)), min(frame_width, max(x1, x2))
            y_min, y_max = max(0, min(y1, y2)), min(frame_height, max(y1, y2))

            # YOLO KIÉRTÉKELÉS
            if run_yolo:
                crop = frame[y_min:y_max, x_min:x_max]
                if crop.size > 0:
                    results = model(crop, conf=conf_threshold, verbose=False)

                    best_class_name = "empty"
                    highest_conf = 0.0

                    if len(results[0].boxes):
                        best_box = sorted(results[0].boxes, key=lambda x: x.conf[0].item(), reverse=True)[0]
                        highest_conf = best_box.conf[0].item()
                        best_class_name = model.names[int(best_box.cls[0].item())]

                    color = colors.get(best_class_name, (255, 255, 255))
                    
                    # Eltároljuk a friss eredményt a memóriában
                    last_predictions[roi_name] = (best_class_name, highest_conf, color)

            # RAJZOLÁS
            if roi_name in last_predictions:
                best_class_name, highest_conf, color = last_predictions[roi_name]
                
                cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), color, 4)
                
                label = f"{roi_name}: {best_class_name} ({highest_conf:.2f})" if highest_conf > 0 else f"{roi_name}: Unrecognized"
                cv2.putText(frame, label, (x_min + 5, y_max - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

        # Az elkészült, bedobozolt képkockát hozzáírjuk az új videóhoz
        video_writer.write(frame)

    # Erőforrások elengedése
    video_capture.release()
    video_writer.release()
    cv2.destroyAllWindows()
    print(f"Az új fix rekeszes videó elmentve: {output_path}\n")

## A következő cellában lehet manuális dobozokat megadni a kód mellé

### Használata:
- A kép bemeneti/kimeneti útvonalát az input_path és output_path változóhoz kell beírni
- A rekeszek formátuma: "Rekesznév": (x1, y1, x2, y2) (pl. "a1": (640, 1160, 1145, 1000))

In [ ]:
if __name__ == "__main__":
    
    # ITT DEFINIÁLJUK A FIX REKESZEKET! 
    # Formátum: 
    box_map = {
        "a1": (640, 1160, 1145, 1000),
        "a2": (1225, 1170, 1710, 1010),
        "a3": (1780, 1190, 2260, 1020),
        "a4": (2310, 1200, 2770, 1030),
        "a5": (2820, 1210, 3250, 1050),
        "a6": (3320, 1220, 3700, 1060),

        "b1": (640, 1340, 1145, 1210),
        "b2": (1225, 1350, 1710, 1210),
        "b3": (1780, 1360, 2260, 1210),
        "b4": (2310, 1360, 2770, 1210),
        "b5": (2820, 1360, 3250, 1240),
        "b6": (3320, 1370, 3700, 1240),

        "c1": (640, 1510, 1145, 1400),
        "c2": (1225, 1520, 1710, 1400),
        "c3": (1780, 1520, 2260, 1400),
        "c4": (2310, 1520, 2770, 1400),
        "c5": (2820, 1520, 3250, 1400),
        "c6": (3320, 1530, 3700, 1400),

        "d1":(640, 1715, 1145, 1560),
        "d2":(1225, 1715, 1710, 1560),
        "d3":(1780, 1715, 2260, 1560),
        "d4":(2310, 1715, 2770, 1560),
        "d5":(2820, 1715, 3250, 1560),
        "d6":(3320, 1715, 3700, 1560),

        "e1":(640, 1940, 1145, 1750),
        "e2":(1225, 1930, 1710, 1750),
        "e3":(1780, 1920, 2260, 1750),
        "e4":(2310, 1910, 2770, 1750),
        "e5":(2820, 1910, 3250, 1750),
        "e6":(3320, 1910, 3700, 1750)
    }

    process_fixed_rois(
        input_path="imgs/3.jpg",
        output_path="imgs/outputs/3ai_f_conf0.png", 
        rois=box_map, 
        model_path="./models/best5.pt",
        conf_threshold=0.26
    )

In [ ]:
# Globális változók az egeres kijelöléshez
rois_dict = {}
points = []
roi_count = 1
clone = None

def draw_roi(event, x, y, flags, param):
    global points, roi_count, clone, rois_dict

    # Balklikk lenyomása
    if event == cv2.EVENT_LBUTTONDOWN:
        points = [(x, y)]

    # Balklikk felengedése
    elif event == cv2.EVENT_LBUTTONUP:
        points.append((x, y))
        
        x1, y1 = points[0]
        x2, y2 = points[1]
        
        # Ha elég a távolság, elmentjük
        if abs(x1 - x2) > 5 and abs(y1 - y2) > 5:
            roi_name = f"CB_{roi_count}"
            rois_dict[roi_name] = (x1, y1, x2, y2)

            # Bal felső saroktül kezdünk
            x_min, y_min = min(x1, x2), min(y1, y2)
            x_max, y_max = max(x1, x2), max(y1, y2)
            
            # Rajzolás
            cv2.rectangle(clone, (x_min, y_min), (x_max, y_max), (0, 255, 0), 8)
            cv2.putText(clone, roi_name, (x_min + 10, y_max - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            cv2.imshow("Boxing Window", clone)
            
            print(f"\"{roi_name}\": {(x1, y1, x2, y2)},")
            roi_count += 1

In [ ]:
def select_rois_manually(image_path):
    global clone, rois_dict, roi_count
    
    rois_dict = {}
    roi_count = 1
    
    image = cv2.imread(image_path)
    if image is None:
        print(f"Hiba: Nem található a kép: {image_path}")
        return {}
        
    clone = image.copy()
    
    # Ablak létrehozása
    cv2.namedWindow("Boxing Window", cv2.WINDOW_NORMAL)
    cv2.setMouseCallback("Boxing Window", draw_roi)
    
    print(f"Boxing Map:")
    while True:
        cv2.imshow("Boxing Window", clone)
        key = cv2.waitKey(1) & 0xFF
        
        # Kilépés 'Enter' vagy ESC (27) gombbal
        if key == 13 or key == 27 or cv2.getWindowProperty("Boxing Window", cv2.WND_PROP_VISIBLE) < 1:
            break
            
    cv2.destroyAllWindows()
    return rois_dict

## A következő cellában leeht elindítani a dobozoló programot

### Használata:
- A kép bemeneti/kimeneti útvonalát az input_path és output_path változóhoz kell beírni
- A program futtatásakor BALKLIKK-et lenyomva tartandó, a kijelölendő doboz egyik sarkától az átellenesig
- A program a sikeres dobozoláshoz rajzol ey zöld dobozt, kiírja a doboz koordinátáit
- A programból az Enter cagy Escape billentyűvel lehet kilépni
- Kilépés befejezése után a program kiértékeli a kapott dobozokat, elmenti az új képet

In [ ]:
# ---------------------------------------------------------
# FŐ FUTTATHATÓ RÉSZ
# ---------------------------------------------------------
if __name__ == "__main__":
    input_img = "imgs/f1.png"
    output_img = "imgs/outputs/f1b.png"
    
    box_map = select_rois_manually(input_img)
    
    # YOLO futtatása a kézzel kijelölt rekeszekre
    if len(box_map) > 0:
        print("\n--- Kijelölés befejezve. YOLO feldolgozás indítása... ---")
        process_fixed_rois(
            input_path=input_img,
            output_path=output_img, 
            rois=box_map,
            model_path="./models/best5.pt",
            conf_threshold=0.01
        )
    else:
        print("\nNem jelöltél ki egyetlen rekeszt sem!")

In [ ]:
if __name__ == "__main__":
    
    # 1. Hard code-olt dobozok
    box_map = {
        "CB_1": (760, 146, 930, 220),
        "CB_2": (1089, 210, 933, 126),
        "CB_3": (1085, 111, 1250, 204),
        "CB_4": (783, 270, 943, 335),
        "CB_5": (949, 263, 1098, 328),
        "CB_6": (1093, 251, 1247, 322),
        "CB_7": (809, 383, 960, 441),
        "CB_8": (950, 370, 1098, 432),
        "CB_9": (1094, 363, 1235, 423),
        "CB_10": (1103, 462, 1232, 517),
        "CB_11": (843, 470, 974, 528),
        "CB_12": (975, 467, 1100, 518),
    }

    # 2. Elérési utak
    input_video_path = "videos/f1.mp4" 
    output_video_path = "videos/outputs/f1bLOC.mp4"

    print("Videó feldolgozásának indítása...")
    
    process_video_fixed_rois(
        input_path=input_video_path,
        output_path=output_video_path,
        rois=box_map,
        model_path="./models/best5.pt",
        frame_proc=20,
        conf_threshold=0.001
    )